# Census Income Prediction — ML Model

## 1. Project Overview
**Goal**: Predict whether an individual's annual income exceeds **$50K** (`>50K`) or is at/below **$50K** (`<=50K`) using demographic and employment data from the US Census (Adult dataset).

### Project Architecture:
1. **Phase 1 (Model Training)**: Data preparation, preprocessing pipeline, model training (Random Forest & Logistic Regression), metric evaluation, and model serialization.
2. **Phase 2 (Backend)**: Expose a REST API using FastAPI for single and batch predictions.
3. **Phase 3 (Frontend)**: Interactive React dashboard for user inputs and real-time visualization.

## 2. Import Libraries
We import standard machine learning and data processing libraries.

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Load Dataset
We load the Adult Census Income dataset (version 2) directly from OpenML.

In [2]:
print("Fetching Adult dataset from OpenML...")
adult = fetch_openml(name='adult', version=2, as_frame=True)
df = adult.frame.copy()

print('Dataset Shape:', df.shape)
df.head()

Fetching Adult dataset from OpenML...
Dataset Shape: (48842, 15)


## 4. Dataset Inspection
Inspect data types, column names, missing values, and numerical statistics.

In [3]:
print('Columns:')
print(df.columns.tolist())

print('\nMissing values:')
missing = df.isna().sum()
print(missing[missing > 0])

print('\nSummary Statistics:')
df.describe()

Columns:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'class']

Missing values:
occupation        2809
workclass         2799
native-country     857
dtype: int64


## 5. Exploratory Data Analysis
Clean the target column and inspect the distribution between `<=50K` and `>50K`.

In [4]:
# Standardize target values by stripping potential trailing dots
df['class'] = df['class'].astype(str).str.replace('.', '', regex=False)

print('Target Distribution:')
print(df['class'].value_counts())

print('\nTarget Percentage:')
print(df['class'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='class', hue='class', palette='Blues', legend=False)
plt.title('Income Class Distribution')
plt.xlabel('Income Class')
plt.ylabel('Count')
plt.show()

Target Distribution:
class
<=50K    37155
>50K     11687
Name: count, dtype: int64

Target Percentage:
class
<=50K    76.071823
>50K     23.928177
Name: proportion, dtype: float64


## 6. Data Preprocessing
We separate features into **numerical** and **categorical** columns:
- **Numerical pipeline**: Impute missing with `median` + scale with `StandardScaler`.
- **Categorical pipeline**: Impute missing with `most_frequent` + encode with `OneHotEncoder(handle_unknown='ignore')`.

All preprocessing is encapsulated in a `ColumnTransformer` to guarantee **no data leakage**.

In [5]:
X = df.drop(columns=['class'])
y = df['class']

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print(f"Numerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Preprocessor built successfully!')

Numerical features (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features (8): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


## 7. Train/Test Split
We perform an **80/20 Stratified Train/Test Split** to preserve the proportion of both income classes.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} ({(len(X_train)/len(X))*100:.1f}%)")
print(f"Testing samples: {len(X_test)} ({(len(X_test)/len(X))*100:.1f}%)")

Training samples: 39073 (80.0%)
Testing samples: 9769 (20.0%)


## 8. Train Logistic Regression
We fit a Logistic Regression pipeline as a linear classification baseline.

In [7]:
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

logistic_pipeline.fit(X_train, y_train)
logistic_pred = logistic_pipeline.predict(X_test)
logistic_acc = accuracy_score(y_test, logistic_pred)

print(f"Logistic Regression Accuracy: {logistic_acc:.4f} ({logistic_acc*100:.2f}%)")

Logistic Regression Accuracy: 0.8524 (85.24%)


## 9. Train Random Forest Classifier
We train a Random Forest ensemble model with 50 decision trees.

In [8]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)")

Random Forest Accuracy: 0.8576 (85.76%)


## 10. Model Comparison
Compare test accuracies to select the best performing model.

In [9]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression'],
    'Accuracy': [rf_acc, logistic_acc],
    'Accuracy (%)': [f"{rf_acc*100:.2f}%", f"{logistic_acc*100:.2f}%"]
})

print(results.to_string(index=False))
print("\n-> Random Forest is selected for the production pipeline.")

                 Model  Accuracy Accuracy (%)
0        Random Forest  0.857611       85.76%
1  Logistic Regression  0.852390       85.24%

-> Random Forest is selected for the production pipeline.


## 11. Model Evaluation
Detailed classification report (Precision, Recall, F1-Score) for the selected Random Forest model.

In [10]:
print('Classification Report (Random Forest):\n')
print(classification_report(y_test, rf_pred, target_names=['<=50K', '>50K']))

Classification Report (Random Forest):

              precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      7431
        >50K       0.74      0.63      0.68      2338

    accuracy                           0.86      9769
   macro avg       0.81      0.78      0.79      9769
weighted avg       0.85      0.86      0.85      9769



## 12. Confusion Matrix
Plotting the Confusion Matrix to inspect true positives, false positives, true negatives, and false negatives.

In [11]:
cm = confusion_matrix(y_test, rf_pred, labels=['<=50K', '>50K'])
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['<=50K', '>50K'],
            yticklabels=['<=50K', '>50K'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()

Confusion Matrix:
[[6905  526]
 [ 865 1473]]


## 13. Save Final Model
We save the entire preprocessing + Random Forest pipeline as `census_income_model.pkl`.

In [12]:
joblib.dump(rf_pipeline, 'census_income_model.pkl')
print('Saved model pipeline: census_income_model.pkl')

Saved model pipeline: census_income_model.pkl


## 14. Generate Metrics Metadata
We export `model_metrics.json` and `dataset_info.json` for consumption by the FastAPI backend and React frontend.

In [13]:
labels = ['<=50K', '>50K']
report_dict = classification_report(y_test, rf_pred, target_names=labels, output_dict=True)

metrics = {
    "selected_model": "Random Forest Classifier",
    "accuracy": round(float(rf_acc), 6),
    "precision_macro": round(float(precision_score(y_test, rf_pred, average='macro')), 4),
    "recall_macro": round(float(recall_score(y_test, rf_pred, average='macro')), 4),
    "f1_macro": round(float(f1_score(y_test, rf_pred, average='macro')), 4),
    "confusion_matrix": {
        "matrix": cm.tolist(),
        "labels": labels,
        "true_negative": int(cm[0][0]),
        "false_positive": int(cm[0][1]),
        "false_negative": int(cm[1][0]),
        "true_positive": int(cm[1][1])
    },
    "sample_counts": {
        "total_samples": int(len(df)),
        "training_samples": int(len(X_train)),
        "testing_samples": int(len(X_test))
    },
    "target_classes": labels
}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

dataset_info = {
    "dataset_name": "Adult / Census Income Dataset",
    "total_samples": int(len(df)),
    "total_features": int(X.shape[1]),
    "target_column": "class",
    "target_classes": labels,
    "numerical_features": numeric_features,
    "categorical_features": categorical_features
}

with open('dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)

print('Saved metrics and dataset info JSON files!')

Saved metrics and dataset info JSON files!


## 15. Sample Prediction
We load `census_income_model.pkl` and test inference with a raw feature record.

In [14]:
loaded_model = joblib.load('census_income_model.pkl')
sample = X_test.iloc[[0]]
prediction = loaded_model.predict(sample)[0]
probabilities = loaded_model.predict_proba(sample)[0]

print('Sample Input Record:')
for col in sample.columns:
    print(f"{col}: {sample[col].values[0]}")

print(f"\nPredicted Class: {prediction}")
print(f"Class Probabilities: <=50K: {probabilities[0]:.2%}, >50K: {probabilities[1]:.2%}")
print(f"Actual Class: {y_test.iloc[0]}")

Sample Input Record:
age: 54
workclass: Private
fnlwgt: 115602
education: HS-grad
education-num: 9
marital-status: Married-civ-spouse
occupation: Other-service
relationship: Wife
race: Black
sex: Female
capital-gain: 0
capital-loss: 0
hours-per-week: 40
native-country: United-States

Predicted Class: <=50K
Class Probabilities: <=50K: 72.00%, >50K: 28.00%
Actual Class: <=50K
